<a href="https://colab.research.google.com/github/MartaPCastillo/Tesis/blob/main/ARIMA%20Grupo%20M%C3%A9xico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ARIMA Grupo México

#Librerías

In [1]:
!pip install pmdarima

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 11.5 MB/s eta 0:00:00


In [2]:
!pip install arch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 15.2 MB/s eta 0:00:00


In [3]:
import pandas as pd
import yfinance as yf # Para descargar datos de acciones
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_pacf
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error
import math
from pmdarima.arima import auto_arima
from sklearn.metrics import mean_absolute_error
from arch import arch_model
from sklearn.metrics import mean_absolute_percentage_error

#Grupo México

In [4]:
#Obtener datos
df = yf.download('GMEXICOB.MX', start='2024-01-01')

/tmp/ipykernel_505/2329736698.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download('GMEXICOB.MX', start='2024-01-01')
[*********************100%***********************]  1 of 1 completed


In [5]:
# Eliminar nivel del ticker
df.columns = df.columns.droplevel(1)

#Rendimientos

Vamos a usar los rendimientos logarítmicos para nuestros cálculos

In [6]:
#Obtener datos
precios = df['Close']

In [9]:
print(precios)

Date
2024-01-02     84.612045
2024-01-03     81.479942
2024-01-04     80.067772
2024-01-05     82.376122
2024-01-08     83.154617
                 ...    
2026-06-22    210.360001
2026-06-23    206.100006
2026-06-24    196.639999
2026-06-25    202.990005
2026-06-26    200.000000
Name: Close, Length: 622, dtype: float64


In [7]:
# Método financiero estándar para rendimientos logarítmicos
df['Rendimientos_Log'] = np.log(precios / precios.shift(1))
print(df['Rendimientos_Log'])

Date
2024-01-02         NaN
2024-01-03   -0.037720
2024-01-04   -0.017483
2024-01-05    0.028422
2024-01-08    0.009406
                ...   
2026-06-22    0.013689
2026-06-23   -0.020459
2026-06-24   -0.046987
2026-06-25    0.031782
2026-06-26   -0.014839
Name: Rendimientos_Log, Length: 622, dtype: float64


In [8]:
#Eliminar NaN solo de rendimientos
df.dropna(subset=['Rendimientos_Log'], inplace=True)
print(df['Rendimientos_Log'])

Date
2024-01-03   -0.037720
2024-01-04   -0.017483
2024-01-05    0.028422
2024-01-08    0.009406
2024-01-09   -0.044407
                ...   
2026-06-22    0.013689
2026-06-23   -0.020459
2026-06-24   -0.046987
2026-06-25    0.031782
2026-06-26   -0.014839
Name: Rendimientos_Log, Length: 621, dtype: float64


#ARIMA

In [18]:
#Nuestro parámetros son p=0, d=0 y q=0
#Aplicamos ARIMA con la función que ya trae Python

modelo0 = ARIMA(df['Rendimientos_Log'].dropna(), order=(0,0,0))
resultado0 = modelo0.fit()

print(resultado0.summary())


                               SARIMAX Results                                
Dep. Variable:       Rendimientos_Log   No. Observations:                  621
Model:                          ARIMA   Log Likelihood                1480.900
Date:                Fri, 26 Jun 2026   AIC                          -2957.800
Time:                        20:43:42   BIC                          -2948.937
Sample:                             0   HQIC                         -2954.355
                                - 621                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0014      0.001      1.499      0.134      -0.000       0.003
sigma2         0.0005   1.54e-05     32.323      0.000       0.000       0.001
Ljung-Box (L1) (Q):                   4.81   Jarque-

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


In [19]:
#Hacemos la prueba para saber si necesitamos utilizar GARCH o no
#Si p<0.5 si se requiere, si p>0.5 no se requiere

from statsmodels.stats.diagnostic import het_arch

arch_test = het_arch(df['Rendimientos_Log'].dropna())

print("LM Statistic:", arch_test[0])
print("p-value:", arch_test[1])

LM Statistic: 7.492585291343513
p-value: 0.6782659589287707


In [20]:
#Confirmaremos con Auto ARIMA que el modelo (0,0,0) es el mejor
Arima = auto_arima(df['Rendimientos_Log'])
print(Arima)

 ARIMA(0,0,1)(0,0,0)[0] intercept


In [21]:
#Como no fue el mejor vamos a evaluar más opciones de modelos ARIMA
#El resultad con menos AIC es mejor
for orden in [(0,0,0),(0,0,1)]:
  modelo01 = ARIMA(df['Rendimientos_Log'].dropna(),order=orden)
  resultado01 = modelo.fit()

  print(orden, resultado01.aic)

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


(0, 0, 0) -2960.7650713962466
(0, 0, 1) -2960.7650713962466


In [22]:
modelo1 = ARIMA(df['Rendimientos_Log'].dropna(),
               order=(0,0,1))

resultado1 = modelo1.fit()

print(resultado1.summary())

                               SARIMAX Results                                
Dep. Variable:       Rendimientos_Log   No. Observations:                  621
Model:                 ARIMA(0, 0, 1)   Log Likelihood                1483.383
Date:                Fri, 26 Jun 2026   AIC                          -2960.765
Time:                        20:44:04   BIC                          -2947.471
Sample:                             0   HQIC                         -2955.598
                                - 621                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0014      0.001      1.657      0.097      -0.000       0.003
ma.L1         -0.0906      0.037     -2.442      0.015      -0.163      -0.018
sigma2         0.0005   1.57e-05     31.336      0.0

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


In [23]:
#Hacemos prueba para saber si se requiere GARCH o no
residuos = resultado1.resid

for lag in [5,10,15,20]:
    lm, pvalue, _, _ = het_arch(residuos, nlags=lag)
    print(f"Lag={lag}: p-value={pvalue}")

Lag=5: p-value=0.4405797490493184
Lag=10: p-value=0.7737874720719256
Lag=15: p-value=0.9349552311345289
Lag=20: p-value=0.9786349869117479


In [25]:
#Aunque no parece necesario GARCH lo vamos a utilizar
garch = arch_model(
    residuos,
    mean='Zero',
    vol='GARCH',
    p=1,
    q=1
)

resultado_garch = garch.fit()
print(resultado_garch.summary())

Iteration:      1,   Func. Count:      5,   Neg. LLF: 54121.772601043354
Iteration:      2,   Func. Count:     11,   Neg. LLF: -1482.8215112761063
Iteration:      3,   Func. Count:     17,   Neg. LLF: -1482.8215112761063
Iteration:      4,   Func. Count:     25,   Neg. LLF: -1024.9812281233935
Iteration:      5,   Func. Count:     33,   Neg. LLF: -1251.0965070321636
Iteration:      6,   Func. Count:     40,   Neg. LLF: -1482.6473838523762
Iteration:      7,   Func. Count:     46,   Neg. LLF: -1082.0152560352053
Iteration:      8,   Func. Count:     52,   Neg. LLF: -1489.0238514589046
Optimization terminated successfully    (Exit mode 0)
            Current function value: -1490.5622783298413
            Iterations: 8
            Function evaluations: 62
            Gradient evaluations: 8
                       Zero Mean - GARCH Model Results                        
Dep. Variable:                   None   R-squared:                       0.000
Mean Model:                 Zero Mean   Ad

/usr/local/lib/python3.12/dist-packages/arch/univariate/base.py:694: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.0004929. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 100 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  self._check_scale(resids)


#Predicción de Precios

In [26]:
#Predicciones pero de los rendimientos log

predictions = resultado1.forecast(steps=5)  # Predict next 5 points
print(predictions)

621    0.002645
622    0.001387
623    0.001387
624    0.001387
625    0.001387
Name: predicted_mean, dtype: float64


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


In [27]:
#Vamos a convertir los rendimientos en precios
ultimo_precio = precios.iloc[-1]   # Último precio conocido

precios_pred = []
precio_actual = ultimo_precio

for r in predictions:
    precio_actual = precio_actual * np.exp(r)
    precios_pred.append(precio_actual)

In [28]:
precios_pred = pd.Series(precios_pred, index=predictions.index)

print(precios_pred)

621    200.529646
622    200.808011
623    201.086762
624    201.365900
625    201.645425
dtype: float64


#RMSE

In [30]:
precios_reales = precios[-5:]
print(precios_reales)

Date
2026-06-22    210.360001
2026-06-23    206.100006
2026-06-24    196.639999
2026-06-25    202.990005
2026-06-26    200.000000
Name: Close, dtype: float64


In [33]:
rmse = np.sqrt(mean_squared_error(precios_reales, precios_pred))
print(f"RMSE = {rmse:.2f}")

RMSE = 5.47


#MAE

In [34]:
mae = mean_absolute_error(precios_reales, precios_pred)
print(f"MAE = {mae: .2f}")

MAE =  4.57


#MAPE

In [38]:
# Using scikit-learn
from sklearn.metrics import mean_absolute_percentage_error

mape = mean_absolute_percentage_error(precios_reales, precios_pred) * 100
print(f"MAPE = {mape: .2f} %")

MAPE =  2.22 %
